# 11h — B4.M.7: Balanço CO₂eq do RenovaBio (queima vs cana_direto)

**Pré-registro v2.5 K1 + revisão para v2.6**

**Objetivo:** quantificar se o RenovaBio **reduz** emissões em GEE líquidos, dado o achado central:
- `log1p_queima`: ATT_INCONCLUSIVO (mas direção pós-2020 sugere aumento, mistura com fenômenos externos)
- `asinh_cana_direto`: ATT_ROBUSTO positivo (mecanização via Eqs. 62-66 SEEG)

**Princípio metodológico (K1 v2.5):** o SEEG já reporta tudo em tCO₂e usando GWP-AR5 padrão (CH4=28, N2O=265). Para o paper, vamos **manter SEEG nativo (AR5)** como cálculo principal — assim os números são diretamente comparáveis com os relatórios oficiais SEEG. Apresentaremos **AR6 como robustez** (CH4=27, N2O=273), seguindo recomendação editorial de *Ecological Economics*.

**Não precisamos reaplicar GWP — apenas somar/subtrair em tCO₂e.**

**Cálculo:**

Para cada município tratado e cada ano t pós-tratamento:

$$\Delta 	ext{GEE}^{liq}_{i,t} = \widehat{ATT}_{	ext{queima}} \cdot 	ext{baseline\_queima}_i + \widehat{ATT}_{	ext{cana\_direto}} \cdot 	ext{baseline\_cana\_direto}_i$$

Convertendo os ATTs do espaço asinh/log1p para escala bruta tCO₂e:

- ATT em log1p de Y → multiplicador (exp(ATT) − 1) sobre baseline Y
- ATT em asinh de Y → para valores moderados, asinh ≈ log; usar mesma fórmula como aproximação

**Saídas:**
- Tabela do balanço médio anual por município tratado (em tCO₂e)
- Variantes AR5 (principal) e AR6 (robustez)
- Decomposição: contribuição relativa de queima vs cana_direto vs fert_n vs calagem

**Pré-condições:**
- `data/interim/panel_canavieiro_main.csv`, `seeg_subcanais_panel.csv`
- `data/interim/att_t1_main.csv` (ATTs macro v2.3.7)
- `data/interim/att_canais_main.csv` (ATTs B4.M.4 v2.4)

## Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/Renovabio - EcoEco")
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
from pipeline.config import interim, out_pre

# Fatores GWP IPCC AR5 (padrão SEEG) e AR6 (robustez)
GWP_AR5 = {"CO2": 1, "CH4": 28, "N2O": 265}
GWP_AR6 = {"CO2": 1, "CH4": 27, "N2O": 273}

print("Fatores GWP:")
print(f"  AR5 (padrao SEEG): CH4={GWP_AR5['CH4']}, N2O={GWP_AR5['N2O']}")
print(f"  AR6 (robustez)   : CH4={GWP_AR6['CH4']}, N2O={GWP_AR6['N2O']}")
print()
print("Como SEEG ja reporta em CO2e-AR5, nao precisamos reaplicar GWP.")
print("AR6 sera calculado por razao: CO2e_AR6 = CO2e_AR5 * (GWP_AR6/GWP_AR5)")
print(f"  Razao N2O: {GWP_AR6['N2O']/GWP_AR5['N2O']:.4f}")
print(f"  Razao CH4: {GWP_AR6['CH4']/GWP_AR5['CH4']:.4f}")

Fatores GWP:
  AR5 (padrao SEEG): CH4=28, N2O=265
  AR6 (robustez)   : CH4=27, N2O=273

Como SEEG ja reporta em CO2e-AR5, nao precisamos reaplicar GWP.
AR6 sera calculado por razao: CO2e_AR6 = CO2e_AR5 * (GWP_AR6/GWP_AR5)
  Razao N2O: 1.0302
  Razao CH4: 0.9643


## Bloco 1 — Carregar painel e ATTs estimados

In [3]:
panel = pd.read_csv(interim("panel_canavieiro_main.csv"), dtype={"geocode": str})
print(f"painel canonico: {panel.shape}")

# Sub-canais SEEG (niveis brutos tCO2e)
seeg_sub = pd.read_csv(interim("seeg_subcanais_panel.csv"))
seeg_sub["geocode"] = seeg_sub["geocode"].astype(str).str.zfill(7)
print(f"seeg sub-canais: {seeg_sub.shape}")

# ATTs macro v2.3.7 (queima vem daqui)
att_macro = pd.read_csv(interim("att_t1_main.csv"))
print(f"att_t1_main (v2.3.7): {att_macro.shape}")

# ATTs B4.M.4 v2.4 (cana_direto, fert_n, calagem vem daqui)
att_canais = pd.read_csv(interim("att_canais_main.csv"))
print(f"att_canais_main (B4.M.4 v2.4): {att_canais.shape}")

painel canonico: (8420, 139)
seeg sub-canais: (23630, 19)
att_t1_main (v2.3.7): (30, 8)
att_canais_main (B4.M.4 v2.4): (16, 8)


## Bloco 2 — Extrair ATTs sob FULL2 (spec principal v2.4)

In [4]:
def extrair_att(df, outcome_name, spec="FULL2"):
    """Extrai (ATT, SE) de df padrao 11a/11d para spec especifica."""
    sub = df[(df["outcome"] == outcome_name) & (df["spec"] == spec)]
    if sub.empty:
        # tentar com outcome ligeiramente diferente
        candidates = df[df["spec"] == spec]["outcome"].unique()
        for c in candidates:
            if outcome_name in c or c in outcome_name:
                sub = df[(df["outcome"] == c) & (df["spec"] == spec)]
                break
    if sub.empty:
        return np.nan, np.nan
    att = float(sub.iloc[0]["ATT"])
    se = float(sub.iloc[0]["SE"]) if pd.notna(sub.iloc[0]["SE"]) else np.nan
    return att, se

# ATTs primarios sob FULL2
att_queima, se_queima = extrair_att(att_macro, "log1p_queima")
att_cana_dir, se_cana_dir = extrair_att(att_canais, "asinh_cana_direto")
att_fert_n, se_fert_n = extrair_att(att_canais, "log1p_fert_n")
att_calagem, se_calagem = extrair_att(att_canais, "log1p_calagem")
att_res_outros, se_res_outros = extrair_att(att_canais, "log1p_res_outros")

print("ATTs sob FULL2 (escala transformada):")
print(f"  queima         : ATT = {att_queima:+.4f} (SE={se_queima:.4f})")
print(f"  cana_direto    : ATT = {att_cana_dir:+.4f} (SE={se_cana_dir:.4f})")
print(f"  fert_n         : ATT = {att_fert_n:+.4f} (SE={se_fert_n:.4f})")
print(f"  calagem        : ATT = {att_calagem:+.4f} (SE={se_calagem:.4f})")
print(f"  res_outros     : ATT = {att_res_outros:+.4f} (SE={se_res_outros:.4f})")

ATTs sob FULL2 (escala transformada):
  queima         : ATT = +0.2052 (SE=0.1435)
  cana_direto    : ATT = +0.2100 (SE=0.0904)
  fert_n         : ATT = +0.1238 (SE=0.0481)
  calagem        : ATT = +0.0381 (SE=0.0184)
  res_outros     : ATT = -0.0572 (SE=0.0267)


## Bloco 3 — Baseline em tCO₂e por município tratado (média 2015-2017)

Calculamos a emissão **média anual pré-tratamento** de cada canal, por município tratado, usando os 3 anos pré-coorte mais antiga (2015-2017). Isto é o "valor X" sobre o qual o ATT (em log-escala) opera.

In [5]:
# Pegar baseline pre-tratamento (2015-2017) dos canais relevantes
# Para queima: usar coluna 'queima' do painel canonico
# Para cana_direto/fert_n/calagem/res_outros: usar seeg_sub

# 1. Identificar canavieiros tratados
tratados = panel[panel["is_treated_ever"] == True]["geocode"].unique()
print(f"Canavieiros tratados: {len(tratados)}")

# 2. Baseline queima (do painel canonico, ja em tCO2e)
baseline_queima = (
    panel[(panel["geocode"].isin(tratados)) & (panel["ano"].between(2015, 2017))]
    .groupby("geocode")["queima"].mean()
)
print(f"\nBaseline queima (media 2015-2017 por muni):")
print(f"  n munis com baseline: {baseline_queima.notna().sum()}")
print(f"  mediana: {baseline_queima.median():.1f} tCO2e/ano")
print(f"  media:   {baseline_queima.mean():.1f} tCO2e/ano")
print(f"  total tratados/ano (soma): {baseline_queima.sum():.0f} tCO2e")

Canavieiros tratados: 194

Baseline queima (media 2015-2017 por muni):
  n munis com baseline: 194
  mediana: 410.9 tCO2e/ano
  media:   520.9 tCO2e/ano
  total tratados/ano (soma): 101060 tCO2e


In [6]:
# 3. Baseline dos sub-canais SEEG
seeg_pre = seeg_sub[
    (seeg_sub["geocode"].isin(tratados))
    & (seeg_sub["ano"].between(2015, 2017))
]

# cana_direto = res_cana + org_cana (consolidacao v2.4)
seeg_pre = seeg_pre.copy()
seeg_pre["cana_direto"] = (seeg_pre["res_cana"].fillna(0)
                           + seeg_pre["org_cana"].fillna(0))

baseline_canais = seeg_pre.groupby("geocode")[
    ["cana_direto", "fert_n", "calagem", "res_outros"]
].mean()
print(f"Baseline sub-canais (media 2015-2017 por muni, tCO2e/ano):")
for col in ["cana_direto", "fert_n", "calagem", "res_outros"]:
    print(f"  {col:15s} mediana={baseline_canais[col].median():.1f}  "
          f"media={baseline_canais[col].mean():.1f}  "
          f"total={baseline_canais[col].sum():.0f}")

Baseline sub-canais (media 2015-2017 por muni, tCO2e/ano):
  cana_direto     mediana=11340.5  media=14734.7  total=2858525
  fert_n          mediana=18140.0  media=38971.8  total=7560530
  calagem         mediana=5860.9  media=13152.4  total=2551571
  res_outros      mediana=3279.1  media=13975.3  total=2711213


## Bloco 4 — Converter ATTs para efeito absoluto em tCO₂e

**Conversão (princípio):**
- ATT em log1p de Y → fator multiplicativo aproximado: $\Delta Y \approx Y_{baseline} \cdot (e^{ATT} - 1)$
- ATT em asinh de Y → para Y moderado a grande, asinh ≈ log; mesma fórmula

Para ATT pequeno, $(e^{ATT}-1) \approx ATT$. Para ATT=0,21, multiplicador é 0,234 (≈ +23%).

In [7]:
def att_to_multiplier(att):
    """Converte ATT em log/asinh para multiplicador (Y_post/Y_pre - 1)."""
    if not np.isfinite(att):
        return np.nan
    return float(np.exp(att) - 1)

# Multiplicadores
mult_queima = att_to_multiplier(att_queima)
mult_cana_dir = att_to_multiplier(att_cana_dir)
mult_fert_n = att_to_multiplier(att_fert_n)
mult_calagem = att_to_multiplier(att_calagem)
mult_res_outros = att_to_multiplier(att_res_outros)

print("Multiplicadores (Y_post / Y_pre - 1):")
print(f"  queima         : {mult_queima:+.4f}  ({mult_queima*100:+.1f}%)")
print(f"  cana_direto    : {mult_cana_dir:+.4f}  ({mult_cana_dir*100:+.1f}%)")
print(f"  fert_n         : {mult_fert_n:+.4f}  ({mult_fert_n*100:+.1f}%)")
print(f"  calagem        : {mult_calagem:+.4f}  ({mult_calagem*100:+.1f}%)")
print(f"  res_outros     : {mult_res_outros:+.4f}  ({mult_res_outros*100:+.1f}%)")
print()

# Aplicar aos baselines: delta em tCO2e por municipio-ano
delta_queima = baseline_queima * mult_queima
delta_cana_dir = baseline_canais["cana_direto"] * mult_cana_dir
delta_fert_n = baseline_canais["fert_n"] * mult_fert_n
delta_calagem = baseline_canais["calagem"] * mult_calagem
delta_res_outros = baseline_canais["res_outros"] * mult_res_outros

print(f"Delta absoluto medio anual por municipio tratado (tCO2e/ano, AR5):")
print(f"  queima         : {delta_queima.mean():+.1f}")
print(f"  cana_direto    : {delta_cana_dir.mean():+.1f}")
print(f"  fert_n         : {delta_fert_n.mean():+.1f}")
print(f"  calagem        : {delta_calagem.mean():+.1f}")
print(f"  res_outros     : {delta_res_outros.mean():+.1f}")

Multiplicadores (Y_post / Y_pre - 1):
  queima         : +0.2278  (+22.8%)
  cana_direto    : +0.2336  (+23.4%)
  fert_n         : +0.1317  (+13.2%)
  calagem        : +0.0388  (+3.9%)
  res_outros     : -0.0556  (-5.6%)

Delta absoluto medio anual por municipio tratado (tCO2e/ano, AR5):
  queima         : +118.7
  cana_direto    : +3442.4
  fert_n         : +5134.1
  calagem        : +510.6
  res_outros     : -777.4


## Bloco 5 — Balanço líquido e síntese

In [8]:
# Balanco liquido: soma das mudancas em todos os canais primarios
# (queima eh INCONCLUSIVO mas reportamos por completude)

# Para o balanco, junta-se as series por geocode
balanco = pd.DataFrame({
    "delta_queima": delta_queima,
    "delta_cana_dir": delta_cana_dir,
    "delta_fert_n": delta_fert_n,
    "delta_calagem": delta_calagem,
    "delta_res_outros": delta_res_outros,
})
balanco["balanco_liquido_total"] = balanco.sum(axis=1)
balanco["balanco_so_robustos"] = (
    balanco["delta_cana_dir"]
    + balanco["delta_fert_n"]
    + balanco["delta_calagem"]
    + balanco["delta_res_outros"]
)  # exclui queima (INCONCLUSIVO)

balanco.to_csv(interim("b4m7_balanco_co2eq.csv"))

print("=" * 70)
print("BALANCO CO2eq POR MUNICIPIO TRATADO (tCO2e/ano, GWP-AR5)")
print("=" * 70)
for col in balanco.columns:
    print(f"  {col:25s} mediana={balanco[col].median():+8.1f}  "
          f"media={balanco[col].mean():+9.1f}  "
          f"total={balanco[col].sum():+12.0f}")

print()
print("=" * 70)
print("BALANCO AGREGADO PARA OS 194 CANAVIEIROS TRATADOS")
print("=" * 70)
n_munis = len(balanco)
total_anual = balanco["balanco_so_robustos"].sum()
print(f"  N municipios: {n_munis}")
print(f"  Balanco liquido anual (canais robustos): {total_anual:+.0f} tCO2e/ano")
print(f"  Media por municipio: {balanco['balanco_so_robustos'].mean():+.1f} tCO2e/ano")
print()
print(f"  Balanco INCLUINDO queima (atencao: INCONCLUSIVO):")
print(f"    Total: {balanco['balanco_liquido_total'].sum():+.0f} tCO2e/ano")
print(f"    Media por municipio: {balanco['balanco_liquido_total'].mean():+.1f} tCO2e/ano")

BALANCO CO2eq POR MUNICIPIO TRATADO (tCO2e/ano, GWP-AR5)
  delta_queima              mediana=   +93.6  media=   +118.7  total=      +23024
  delta_cana_dir            mediana= +2649.4  media=  +3442.4  total=     +667816
  delta_fert_n              mediana= +2389.7  media=  +5134.1  total=     +996015
  delta_calagem             mediana=  +227.5  media=   +510.6  total=      +99056
  delta_res_outros          mediana=  -182.4  media=   -777.4  total=     -150820
  balanco_liquido_total     mediana= +5801.1  media=  +8428.3  total=    +1635091
  balanco_so_robustos       mediana= +5670.2  media=  +8309.6  total=    +1612067

BALANCO AGREGADO PARA OS 194 CANAVIEIROS TRATADOS
  N municipios: 194
  Balanco liquido anual (canais robustos): +1612067 tCO2e/ano
  Media por municipio: +8309.6 tCO2e/ano

  Balanco INCLUINDO queima (atencao: INCONCLUSIVO):
    Total: +1635091 tCO2e/ano
    Media por municipio: +8428.3 tCO2e/ano


## Bloco 6 — Variante AR6 (robustez)

In [9]:
# AR6 ajusta apenas N2O (CH4 quase identico: 27 vs 28)
# Os ATTs ja estao em CO2e-AR5; convertemos para CO2e-AR6 por razao

# Composicao GEE dos canais:
# - queima: ~70% CH4 + ~30% N2O (combustao de residuos)
# - cana_direto: 100% N2O (Eqs. 62-66 + 40, 42)
# - fert_n: 100% N2O (Eqs. 52-54)
# - calagem: 100% CO2 (sem mudanca AR5 vs AR6)
# - res_outros: 100% N2O (Eqs. 60-61)

razao_n2o = GWP_AR6["N2O"] / GWP_AR5["N2O"]
razao_ch4 = GWP_AR6["CH4"] / GWP_AR5["CH4"]
razao_co2 = 1.0

# Composicao simplificada (assumida com base nas Eqs. SEEG)
composicao = {
    "delta_queima": {"CH4": 0.70, "N2O": 0.30},
    "delta_cana_dir": {"N2O": 1.0},
    "delta_fert_n": {"N2O": 1.0},
    "delta_calagem": {"CO2": 1.0},
    "delta_res_outros": {"N2O": 1.0},
}

balanco_ar6 = pd.DataFrame()
for col, comp in composicao.items():
    fator = (comp.get("CO2", 0) * razao_co2 + comp.get("CH4", 0) * razao_ch4
             + comp.get("N2O", 0) * razao_n2o)
    balanco_ar6[col + "_ar6"] = balanco[col] * fator

balanco_ar6["balanco_so_robustos_ar6"] = (
    balanco_ar6["delta_cana_dir_ar6"]
    + balanco_ar6["delta_fert_n_ar6"]
    + balanco_ar6["delta_calagem_ar6"]
    + balanco_ar6["delta_res_outros_ar6"]
)
balanco_ar6["balanco_liquido_total_ar6"] = balanco_ar6.iloc[:, :5].sum(axis=1)

balanco_ar6.to_csv(interim("b4m7_balanco_co2eq_ar6.csv"))

print("=" * 70)
print("BALANCO CO2eq POR MUNICIPIO TRATADO (tCO2e/ano, GWP-AR6) — ROBUSTEZ")
print("=" * 70)
print(f"{'canal':<25}{'mediana AR5':>14}{'mediana AR6':>14}{'diff':>10}")
print("-" * 70)
mapping = [
    ("delta_queima",       "delta_queima_ar6"),
    ("delta_cana_dir",     "delta_cana_dir_ar6"),
    ("delta_fert_n",       "delta_fert_n_ar6"),
    ("delta_calagem",      "delta_calagem_ar6"),
    ("delta_res_outros",   "delta_res_outros_ar6"),
    ("balanco_so_robustos","balanco_so_robustos_ar6"),
]
for c5, c6 in mapping:
    if c5 in balanco.columns and c6 in balanco_ar6.columns:
        m5 = balanco[c5].median()
        m6 = balanco_ar6[c6].median()
        print(f"  {c5:<23}{m5:>+14.1f}{m6:>+14.1f}{m6-m5:>+10.1f}")

BALANCO CO2eq POR MUNICIPIO TRATADO (tCO2e/ano, GWP-AR6) — ROBUSTEZ
canal                       mediana AR5   mediana AR6      diff
----------------------------------------------------------------------
  delta_queima                    +93.6         +92.1      -1.5
  delta_cana_dir                +2649.4       +2729.4     +80.0
  delta_fert_n                  +2389.7       +2461.9     +72.1
  delta_calagem                  +227.5        +227.5      +0.0
  delta_res_outros               -182.4        -187.9      -5.5
  balanco_so_robustos           +5670.2       +5834.8    +164.6


## Bloco 7 — Interpretação editorial

In [10]:
print("=" * 70)
print("INTERPRETACAO EDITORIAL DO BALANCO CO2eq (K1 v2.5)")
print("=" * 70)
print()

bal_robustos = balanco["balanco_so_robustos"].mean()
bal_total = balanco["balanco_liquido_total"].mean()
n_munis = len(balanco)

print(f"Por municipio tratado (medio anual, AR5):")
print(f"  Balanco canais ROBUSTOS: {bal_robustos:+.1f} tCO2e/ano")
print(f"  Balanco TOTAL (incl. queima inconclusiva): {bal_total:+.1f} tCO2e/ano")
print()
print(f"Agregado para 194 canavieiros tratados:")
print(f"  Soma anual (canais robustos): {balanco['balanco_so_robustos'].sum():+.0f} tCO2e/ano")
print(f"  Soma anual (total): {balanco['balanco_liquido_total'].sum():+.0f} tCO2e/ano")
print()

# Leitura editorial
print("LEITURA EDITORIAL (K1 v2.5):")
print()
if bal_robustos > 0:
    print(f"  Os canais ROBUSTOS (cana_direto + fert_n + calagem + res_outros)")
    print(f"  somam efeito liquido POSITIVO de {bal_robustos:+.1f} tCO2e/ano por")
    print(f"  municipio tratado. RenovaBio AUMENTA emissoes via mecanizacao.")
    print()
    print(f"  Decomposicao por canal (% do balanco robustos):")
    for col in ["delta_cana_dir", "delta_fert_n", "delta_calagem", "delta_res_outros"]:
        pct = balanco[col].mean() / bal_robustos * 100 if bal_robustos != 0 else 0
        print(f"    {col:25s} {balanco[col].mean():+8.1f} tCO2e/ano  ({pct:+5.1f}%)")
elif bal_robustos < 0:
    print(f"  Balanco liquido NEGATIVO ({bal_robustos:+.1f} tCO2e/ano por muni).")
    print(f"  RenovaBio REDUZ emissoes em termos liquidos.")

print()
print("OBSERVACAO IMPORTANTE (K1 v2.5):")
print("  Queima e classificada ATT_INCONCLUSIVO no event-study (11f) devido a")
print("  fenomenos confundidores pos-2020 (Operacao Carbono Oculto, queimas PCC")
print("  em SP). Por isso reportamos balanco SEM queima como principal.")
print("  Se H1b queima<0 do v2.3.7 fosse confirmavel causalmente, o balanco")
print("  liquido total seria reduzido pelo ganho da queima (atualmente nao temos")
print("  identificacao limpa para fazer essa afirmacao).")

INTERPRETACAO EDITORIAL DO BALANCO CO2eq (K1 v2.5)

Por municipio tratado (medio anual, AR5):
  Balanco canais ROBUSTOS: +8309.6 tCO2e/ano
  Balanco TOTAL (incl. queima inconclusiva): +8428.3 tCO2e/ano

Agregado para 194 canavieiros tratados:
  Soma anual (canais robustos): +1612067 tCO2e/ano
  Soma anual (total): +1635091 tCO2e/ano

LEITURA EDITORIAL (K1 v2.5):

  Os canais ROBUSTOS (cana_direto + fert_n + calagem + res_outros)
  somam efeito liquido POSITIVO de +8309.6 tCO2e/ano por
  municipio tratado. RenovaBio AUMENTA emissoes via mecanizacao.

  Decomposicao por canal (% do balanco robustos):
    delta_cana_dir             +3442.4 tCO2e/ano  (+41.4%)
    delta_fert_n               +5134.1 tCO2e/ano  (+61.8%)
    delta_calagem               +510.6 tCO2e/ano  ( +6.1%)
    delta_res_outros            -777.4 tCO2e/ano  ( -9.4%)

OBSERVACAO IMPORTANTE (K1 v2.5):
  Queima e classificada ATT_INCONCLUSIVO no event-study (11f) devido a
  fenomenos confundidores pos-2020 (Operacao Carbono 

## Conclusão B4.M.7

Saídas:
- `b4m7_balanco_co2eq.csv` — balanço por município tratado (AR5)
- `b4m7_balanco_co2eq_ar6.csv` — variante AR6 (robustez)

**Interpretação principal:** o balanço dos canais robustos quantifica o efeito líquido em tCO₂e do RenovaBio. Junto com o resultado do event-study de queima (inconclusivo), este número fecha a narrativa K1 v2.5 sobre transferência contábil entre canais SEEG.

**Próximo passo:** consolidar v2.6 do pré-registro com todos os achados (K1 + K2 + K3 + dose-response invertido + balanço CO₂eq + anotações Carbono Oculto/etanol milho/sem desmatamento) e partir para escrita do manuscrito.

In [11]:
# ============================================================================
# B4.M.8 — Intensidade de carbono por tonelada de cana
# Cole no 11h APÓS o Bloco 7 (em memória: balanco, baseline_canais, mult_*,
# panel, tratados, mult_cana_dir, mult_fert_n, mult_calagem, mult_res_outros)
#
# 4 células independentes — separe cada uma como célula nova se quiser.
# ============================================================================

# ========== CÉLULA 1 — Setup e identificação de produção ==========
import numpy as np
import pandas as pd

print("=" * 78)
print("B4.M.8 — INTENSIDADE DE CARBONO por TONELADA DE CANA")
print("=" * 78)
print()
print("Pergunta: emissão por tonelada produzida sobe, fica estável ou cai?")
print("Decomposição Kaya: efeito ESCALA (mais produção) vs INTENSIDADE.")
print()

# Procurar coluna de produção de cana
prod_cols_candidates = [
    'pam_quantidade_cana', 'pam_qtd_cana', 'pam_producao_cana_t',
    'pam_quantidade_produzida_cana', 'producao_cana_t', 'cana_producao_t',
]
prod_col = None
for c in prod_cols_candidates:
    if c in panel.columns:
        prod_col = c
        print(f"OK encontrada coluna de produção: {c}")
        break

if prod_col is None:
    print("AVISO: sem coluna direta de produção de cana.")
    print("Proxy: produção = pam_area_cana_t (ha) x 75 ton/ha (rendimento CS médio)")
    print("Fonte rendimento: CONAB/IBGE 2015-2024 ~70-80 ton/ha em CS")
    if 'pam_area_cana_t' in panel.columns:
        panel['_producao_cana_proxy_t'] = panel['pam_area_cana_t'] * 75
        prod_col = '_producao_cana_proxy_t'
        print(f"Proxy criado em coluna: {prod_col}")
    else:
        raise ValueError("Sem pam_area_cana_t — preciso reler painel com PAM joined")


# ========== CÉLULA 2 — Baseline produção e intensidade pré ==========
baseline_prod = (
    panel[(panel["geocode"].isin(tratados))
          & (panel["ano"].between(2015, 2017))]
    .groupby("geocode")[prod_col].mean()
)
baseline_prod = baseline_prod[baseline_prod > 0]
print(f"Baseline produção cana (média 2015-2017, ton/ano):")
print(f"  n munis com produção > 0: {len(baseline_prod)}")
print(f"  mediana: {baseline_prod.median():,.0f} ton/ano")
print(f"  media:   {baseline_prod.mean():,.0f} ton/ano")
print(f"  total: {baseline_prod.sum():,.0f} ton/ano")
print()

# Intensidade pre-tratamento: kg CO2e / ton cana
total_baseline_canais = baseline_canais.sum(axis=1)  # tCO2e/ano/muni
intensity_pre = (total_baseline_canais * 1000) / baseline_prod  # x1000 -> kg
intensity_pre = intensity_pre[intensity_pre.notna()
                              & np.isfinite(intensity_pre)
                              & (intensity_pre > 0)
                              & (intensity_pre < 5000)]  # filtra outliers absurdos

print(f"INTENSIDADE BASELINE (kg CO2e/ton cana, pré-tratamento):")
print(f"  n munis válidos: {len(intensity_pre)}")
print(f"  mediana: {intensity_pre.median():.2f} kg CO2e/ton")
print(f"  media:   {intensity_pre.mean():.2f} kg CO2e/ton")
print(f"  P25/P75: {intensity_pre.quantile(0.25):.2f} / {intensity_pre.quantile(0.75):.2f}")
print()
print("Referência LCA cana-de-açúcar Brasil (fase agrícola, exclui usina):")
print("  Bordonal et al. 2018, Renew Sust Energy Rev: 20-50 kg CO2e/ton")
print("  Macedo et al. 2008: ~25 kg CO2e/ton (LCA clássica)")
print("  Nossos baselines compatíveis se cair nessa faixa.")


# ========== CÉLULA 3 — Calcular ATT de produção e intensidade pós ==========
try:
    att_substituicao = pd.read_csv(interim("att_substituicao_pam.csv"))
    att_area_cana, _ = extrair_att(att_substituicao, "log1p_pam_area_cana_t",
                                    spec="FULL2")
    mult_area_cana = att_to_multiplier(att_area_cana)
    print(f"ATT em log1p_pam_area_cana_t (FULL2): {att_area_cana:+.4f}")
    print(f"Multiplicador área cana: {mult_area_cana:+.4f} ({mult_area_cana*100:+.1f}%)")
    print()
    print("Premissa: ATT em log(produção) ~ ATT em log(área), assumindo")
    print("rendimento estável. Se rendimento mudou, isto subestima/superestima")
    print("a expansão de produção. Documentar como limitação.")
except Exception as e:
    print(f"Sem ATT de área cana acessível ({e}). Usando mult=0 (cenário pessimista).")
    mult_area_cana = 0.0
print()

# Emissões pós por canal (aplica ATT cada um)
emissoes_pos = pd.DataFrame({
    "cana_direto": baseline_canais["cana_direto"] * (1 + mult_cana_dir),
    "fert_n":      baseline_canais["fert_n"]      * (1 + mult_fert_n),
    "calagem":     baseline_canais["calagem"]     * (1 + mult_calagem),
    "res_outros":  baseline_canais["res_outros"]  * (1 + mult_res_outros),
})
emissoes_pos["total"] = emissoes_pos.sum(axis=1)
producao_pos = baseline_prod * (1 + mult_area_cana)

# Intensidade pos
intensity_pos = (emissoes_pos["total"] * 1000) / producao_pos
intensity_pos = intensity_pos[intensity_pos.notna()
                              & np.isfinite(intensity_pos)
                              & (intensity_pos > 0)
                              & (intensity_pos < 5000)]

common_munis = intensity_pre.index.intersection(intensity_pos.index)
print(f"Munis com intensidade pre+pos válida: {len(common_munis)}")

delta_intensity = intensity_pos.loc[common_munis] - intensity_pre.loc[common_munis]
relative_delta = delta_intensity / intensity_pre.loc[common_munis]


# ========== CÉLULA 4 — Resultados e decomposição Kaya ==========
print("=" * 78)
print("RESULTADO PRINCIPAL — INTENSIDADE DE CARBONO (kg CO2e/ton cana)")
print("=" * 78)
print(f"{'metrica':<38}{'mediana':>12}{'media':>12}")
print("-" * 62)
print(f"{'Intensidade PRE (kg CO2e/ton)':<38}"
      f"{intensity_pre.loc[common_munis].median():>12.2f}"
      f"{intensity_pre.loc[common_munis].mean():>12.2f}")
print(f"{'Intensidade POS (kg CO2e/ton)':<38}"
      f"{intensity_pos.loc[common_munis].median():>12.2f}"
      f"{intensity_pos.loc[common_munis].mean():>12.2f}")
print(f"{'Delta absoluto (kg CO2e/ton)':<38}"
      f"{delta_intensity.median():>+12.2f}"
      f"{delta_intensity.mean():>+12.2f}")
print(f"{'Delta relativo (%)':<38}"
      f"{relative_delta.median()*100:>+11.1f}%"
      f"{relative_delta.mean()*100:>+11.1f}%")
print()
n_caiu = (delta_intensity < 0).sum()
n_subiu = (delta_intensity > 0).sum()
print(f"  Munis com intensidade CAINDO: {n_caiu} ({n_caiu/len(common_munis)*100:.0f}%)")
print(f"  Munis com intensidade SUBINDO: {n_subiu} ({n_subiu/len(common_munis)*100:.0f}%)")
print()

# Decomposição Kaya
emiss_pre = total_baseline_canais.loc[common_munis]
emiss_pos = emissoes_pos["total"].loc[common_munis]
delta_emiss = emiss_pos - emiss_pre
prod_pre = baseline_prod.loc[common_munis]
prod_pos = producao_pos.loc[common_munis]
delta_prod = prod_pos - prod_pre

termo_escala = (intensity_pre.loc[common_munis] / 1000) * delta_prod
termo_intensidade = (delta_intensity / 1000) * prod_pre
termo_interacao = (delta_intensity / 1000) * delta_prod

print("=" * 78)
print("DECOMPOSICAO KAYA — efeito ESCALA vs INTENSIDADE")
print("=" * 78)
print()
print(f"Delta emissões total médio: {delta_emiss.mean():+.0f} tCO2e/muni/ano")
print()
print(f"  Termo ESCALA   (mais produção, intensidade pré):  {termo_escala.mean():+.0f}")
print(f"  Termo INTENS.  (mais emissão/ton, produção pré):  {termo_intensidade.mean():+.0f}")
print(f"  Termo INTERAC. (delta x delta):                   {termo_interacao.mean():+.0f}")
print(f"  Soma (validação): {(termo_escala+termo_intensidade+termo_interacao).mean():+.0f}")
print()

de_mean = delta_emiss.mean()
if abs(de_mean) > 1:
    pct_escala = termo_escala.mean() / de_mean * 100
    pct_int = termo_intensidade.mean() / de_mean * 100
    pct_inter = termo_interacao.mean() / de_mean * 100
    print(f"  % devido a ESCALA:      {pct_escala:.0f}%")
    print(f"  % devido a INTENSIDADE: {pct_int:.0f}%")
    print(f"  % devido a INTERAÇÃO:   {pct_inter:.0f}%")
    print()
    print("LEITURA EDITORIAL:")
    if pct_escala > 70:
        print("  Aumento de emissões é dominado por EXPANSÃO de produção.")
        print("  RenovaBio aumenta cana plantada — mais ton produzidas geram")
        print("  emissão SEEG proporcional. Intensidade por ton ~ ESTÁVEL.")
        print("  Compatível com K1: assinatura contábil sem degradação real.")
    elif pct_int > 70:
        print("  Aumento de emissões é dominado por INTENSIFICAÇÃO real.")
        print("  Cada tonelada de cana emite mais SEEG. RenovaBio AUMENTA")
        print("  a pegada de carbono unitária da cana.")
        print("  Contradiz K1: efeito não é só contábil — há piora real.")
    else:
        print("  Aumento é COMBINAÇÃO de expansão + intensificação.")
        print("  Ambos contribuem para o aumento total observado.")
        print("  Refinamento de K1 necessário: mecanização aumenta intensidade")
        print("  unitária além do efeito de escala.")

# Salvar
b4m8 = pd.DataFrame({
    "intensity_pre_kgco2_ton": intensity_pre.loc[common_munis],
    "intensity_pos_kgco2_ton": intensity_pos.loc[common_munis],
    "delta_intensity_kgco2_ton": delta_intensity,
    "delta_relativo_pct": relative_delta * 100,
    "termo_escala_tco2e": termo_escala,
    "termo_intensidade_tco2e": termo_intensidade,
    "termo_interacao_tco2e": termo_interacao,
})
b4m8.to_csv(interim("b4m8_intensidade_carbono.csv"))
print()
print("OK b4m8_intensidade_carbono.csv salvo")

B4.M.8 — INTENSIDADE DE CARBONO por TONELADA DE CANA

Pergunta: emissão por tonelada produzida sobe, fica estável ou cai?
Decomposição Kaya: efeito ESCALA (mais produção) vs INTENSIDADE.

OK encontrada coluna de produção: pam_qtd_cana
Baseline produção cana (média 2015-2017, ton/ano):
  n munis com produção > 0: 194
  mediana: 1,265,508 ton/ano
  media:   1,645,643 ton/ano
  total: 319,254,787 ton/ano

INTENSIDADE BASELINE (kg CO2e/ton cana, pré-tratamento):
  n munis válidos: 193
  mediana: 26.37 kg CO2e/ton
  media:   57.47 kg CO2e/ton
  P25/P75: 20.94 / 71.86

Referência LCA cana-de-açúcar Brasil (fase agrícola, exclui usina):
  Bordonal et al. 2018, Renew Sust Energy Rev: 20-50 kg CO2e/ton
  Macedo et al. 2008: ~25 kg CO2e/ton (LCA clássica)
  Nossos baselines compatíveis se cair nessa faixa.
ATT em log1p_pam_area_cana_t (FULL2): +0.1897
Multiplicador área cana: +0.2089 (+20.9%)

Premissa: ATT em log(produção) ~ ATT em log(área), assumindo
rendimento estável. Se rendimento mudou, i

In [12]:
# ============================================================================
# B4.M.8 ROBUSTEZ — decomposição por tamanho e coorte
# Cole após o B4.M.8 (em memória: b4m8, baseline_prod, common_munis, panel)
# ============================================================================
import numpy as np
import pandas as pd

print("=" * 78)
print("B4.M.8 ROBUSTEZ — heterogeneidade do achado (intensidade cai 100%?)")
print("=" * 78)

# ============================================================================
# 1. Distribuição completa do delta de intensidade
# ============================================================================
delta = b4m8["delta_intensity_kgco2_ton"]
rel = b4m8["delta_relativo_pct"]

print()
print("1. DISTRIBUIÇÃO DO DELTA DE INTENSIDADE (kg CO2e/ton)")
print(f"   n municípios: {len(delta)}")
print()
print(f"   {'percentil':<12}{'delta abs':>14}{'delta rel %':>14}")
print("   " + "-"*40)
for p in [5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"   P{p:<11}{delta.quantile(p/100):>+14.3f}{rel.quantile(p/100):>+14.2f}")
print()
print(f"   min:     {delta.min():+.3f}  (rel: {rel.min():+.2f}%)")
print(f"   max:     {delta.max():+.3f}  (rel: {rel.max():+.2f}%)")
print()
print("   Quanto da queda fica perto de zero (|rel| < 1%)?")
quase_zero = (rel.abs() < 1.0).sum()
print(f"   munis com |delta rel| < 1%: {quase_zero}/{len(rel)} "
      f"({quase_zero/len(rel)*100:.0f}%)")
print()
print("   Conclusão: se P5 ainda for negativo (não-trivial), achado é robusto.")
print("   Se a maioria está perto de zero, a queda média é pequena na prática.")

# ============================================================================
# 2. Decomposição Kaya por DECIL de produção pre-tratamento
# ============================================================================
print()
print("=" * 78)
print("2. KAYA POR DECIL DE PRODUÇÃO PRE-TRATAMENTO")
print("=" * 78)

# Tamanho do município = baseline_prod
size = baseline_prod.loc[common_munis]
deciles = pd.qcut(size, q=10, labels=[f"D{i+1}" for i in range(10)],
                  duplicates="drop")

b4m8_full = b4m8.copy()
b4m8_full["size_pre"] = size
b4m8_full["size_decil"] = deciles
b4m8_full["intensity_pre"] = b4m8["intensity_pre_kgco2_ton"]
b4m8_full["intensity_pos"] = b4m8["intensity_pos_kgco2_ton"]

por_decil = b4m8_full.groupby("size_decil", observed=True).agg(
    n=("delta_intensity_kgco2_ton", "size"),
    size_min=("size_pre", "min"),
    size_max=("size_pre", "max"),
    int_pre_med=("intensity_pre", "median"),
    int_pos_med=("intensity_pos", "median"),
    delta_med=("delta_intensity_kgco2_ton", "median"),
    delta_rel_med=("delta_relativo_pct", "median"),
    termo_escala_med=("termo_escala_tco2e", "median"),
    termo_int_med=("termo_intensidade_tco2e", "median"),
)

print(f"{'decil':<7}{'n':>4}{'prod min':>11}{'prod max':>12}"
      f"{'int pré':>9}{'int pós':>9}{'Δ abs':>8}{'Δ %':>8}"
      f"{'escala':>10}{'intens.':>10}")
print("-" * 96)
for d, row in por_decil.iterrows():
    print(f"{d:<7}{int(row['n']):>4}"
          f"{row['size_min']:>11,.0f}{row['size_max']:>12,.0f}"
          f"{row['int_pre_med']:>9.2f}{row['int_pos_med']:>9.2f}"
          f"{row['delta_med']:>+8.2f}{row['delta_rel_med']:>+7.1f}%"
          f"{row['termo_escala_med']:>+10,.0f}{row['termo_int_med']:>+10,.0f}")

print()
print("INTERPRETAÇÃO:")
print("- Se D10 (gigantes) puxa toda a queda mediana: efeito concentrado")
print("- Se queda é homogênea entre decis: achado robusto")
print("- Se Termo escala é positivo em todos: expansão acontece em todos")
print("- Se Termo intensidade negativo só em alguns: heterogeneidade real")

# ============================================================================
# 3. Decomposição por COORTE de tratamento
# ============================================================================
print()
print("=" * 78)
print("3. KAYA POR COORTE DE TRATAMENTO (g_m_cs)")
print("=" * 78)

# Pegar coorte de cada município
muni_cohort = (
    panel[(panel["geocode"].isin(common_munis))
          & (panel["g_m_cs"].notna())]
    .groupby("geocode")["g_m_cs"].first()
)
b4m8_full["cohort"] = b4m8_full.index.map(muni_cohort.to_dict())
b4m8_full["cohort"] = b4m8_full["cohort"].astype("Int64")

por_cohort = b4m8_full.groupby("cohort", observed=True).agg(
    n=("delta_intensity_kgco2_ton", "size"),
    int_pre_med=("intensity_pre", "median"),
    int_pos_med=("intensity_pos", "median"),
    delta_med=("delta_intensity_kgco2_ton", "median"),
    delta_rel_med=("delta_relativo_pct", "median"),
    delta_rel_mean=("delta_relativo_pct", "mean"),
    n_caiu=("delta_intensity_kgco2_ton", lambda x: (x < 0).sum()),
    n_subiu=("delta_intensity_kgco2_ton", lambda x: (x > 0).sum()),
)

print(f"{'cohort':>8}{'n':>5}{'int pré':>10}{'int pós':>10}"
      f"{'Δ med':>9}{'Δ rel%':>9}{'caiu':>7}{'subiu':>7}")
print("-" * 65)
for c, row in por_cohort.iterrows():
    print(f"{int(c):>8}{int(row['n']):>5}"
          f"{row['int_pre_med']:>10.2f}{row['int_pos_med']:>10.2f}"
          f"{row['delta_med']:>+9.2f}{row['delta_rel_med']:>+8.1f}%"
          f"{int(row['n_caiu']):>7}{int(row['n_subiu']):>7}")

print()
print("INTERPRETAÇÃO:")
print("- Coorte 2020 (n~142): dominante. Se queda concentra aqui, ok.")
print("- Coortes 2021+: confirmação adicional se sinal mantém-se.")
print("- Se 1 coorte tem todos os munis caindo e outras 50/50: heterog.")

# ============================================================================
# 4. Decomposição por TERCIL share_cana (link com B4.M.5)
# ============================================================================
print()
print("=" * 78)
print("4. KAYA POR TERCIL SHARE-CANA (link com B4.M.5)")
print("=" * 78)

# Pegar tercil de cada município (do painel — coluna foi criada em B4.M.5/11g)
if "tercil_share_cana" in panel.columns:
    muni_tercil = (
        panel[panel["geocode"].isin(common_munis)]
        .groupby("geocode")["tercil_share_cana"].first()
    )
    b4m8_full["tercil"] = b4m8_full.index.map(muni_tercil.to_dict())

    por_tercil = b4m8_full.groupby("tercil", observed=True).agg(
        n=("delta_intensity_kgco2_ton", "size"),
        int_pre_med=("intensity_pre", "median"),
        int_pos_med=("intensity_pos", "median"),
        delta_med=("delta_intensity_kgco2_ton", "median"),
        delta_rel_med=("delta_relativo_pct", "median"),
        n_caiu=("delta_intensity_kgco2_ton", lambda x: (x < 0).sum()),
    )
    print(f"{'tercil':<12}{'n':>5}{'int pré':>10}{'int pós':>10}"
          f"{'Δ med':>9}{'Δ rel%':>9}{'caiu':>7}")
    print("-" * 60)
    for t, row in por_tercil.iterrows():
        print(f"{t:<12}{int(row['n']):>5}"
              f"{row['int_pre_med']:>10.2f}{row['int_pos_med']:>10.2f}"
              f"{row['delta_med']:>+9.2f}{row['delta_rel_med']:>+8.1f}%"
              f"{int(row['n_caiu']):>7}")
    print()
    print("INTERPRETAÇÃO link com B4.M.5:")
    print("- T3 (puros) tem ATT cana_direto NULO. Se T3 ainda mostra queda")
    print("  de intensidade aqui: confirma que a queda vem por expansão,")
    print("  não por ATT mecanístico (porque T3 não tem ATT mecanístico).")
    print("- T2 e T1: queda devida a ambos efeitos (escala + intensificação")
    print("  premiada pelo programa).")
else:
    print("AVISO: tercil_share_cana não está no painel.")
    print("Pulando análise por tercil. Para incluir, rode B4.M.5 antes.")

# Salvar
b4m8_full.to_csv(interim("b4m8_robustez_decomposicao.csv"))
print()
print("OK b4m8_robustez_decomposicao.csv salvo")
print()
print("=" * 78)
print("CONCLUSÃO DA ROBUSTEZ")
print("=" * 78)
print(f"Achado central B4.M.8: intensidade cai em 100% dos munis tratados.")
print()
print(f"Se decis e coortes mostram queda HOMOGÊNEA → achado super-robusto")
print(f"Se um decil/coorte específico domina → reportar com nuance no paper")

B4.M.8 ROBUSTEZ — heterogeneidade do achado (intensidade cai 100%?)

1. DISTRIBUIÇÃO DO DELTA DE INTENSIDADE (kg CO2e/ton)
   n municípios: 193

   percentil        delta abs   delta rel %
   ----------------------------------------
   P5                 -20.305        -13.94
   P10                -11.830        -12.06
   P25                 -5.261         -7.99
   P50                 -1.976         -6.13
   P75                 -0.983         -4.81
   P90                 -0.821         -4.22
   P95                 -0.775         -4.04
   P99                 -0.694         -3.85

   min:     -66.553  (rel: -16.85%)
   max:     -0.630  (rel: -3.65%)

   Quanto da queda fica perto de zero (|rel| < 1%)?
   munis com |delta rel| < 1%: 0/193 (0%)

   Conclusão: se P5 ainda for negativo (não-trivial), achado é robusto.
   Se a maioria está perto de zero, a queda média é pequena na prática.

2. KAYA POR DECIL DE PRODUÇÃO PRE-TRATAMENTO
decil     n   prod min    prod max  int pré  int pós   Δ a

KeyError: 'g_m_cs'

In [13]:
# ============================================================================
# HOTFIX B4.M.8 — coorte e tercil robustos
# Cole após o erro. Reusa b4m8_full já criado na seção 2.
# ============================================================================

import numpy as np
import pandas as pd

print("=" * 78)
print("HOTFIX — localizar coluna de coorte e rodar KAYA por coorte")
print("=" * 78)

# ----------------------------------------------------------------------------
# 1. Função auxiliar para localizar coluna municipal em DataFrames
# ----------------------------------------------------------------------------

def _find_geocode_col(df):
    candidates = ["geocode", "cod_mun", "codigo_municipio", "CD_MUN", "id_municipio"]
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _find_first_valid_source(attr_candidates, required_geo=True):
    """
    Procura em DataFrames na memória uma coluna candidata, junto com geocode.
    Retorna: source_name, dataframe, geocode_col, attr_col
    """
    preferred_names = ["panel_cs", "panel", "data", "data_test", "df", "base"]

    # Primeiro tenta nomes preferenciais
    for name in preferred_names:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            obj = globals()[name]
            geo_col = _find_geocode_col(obj)
            if required_geo and geo_col is None:
                continue
            for attr in attr_candidates:
                if attr in obj.columns:
                    return name, obj, geo_col, attr

    # Depois varre todos os DataFrames em memória
    for name, obj in globals().items():
        if isinstance(obj, pd.DataFrame):
            geo_col = _find_geocode_col(obj)
            if required_geo and geo_col is None:
                continue
            for attr in attr_candidates:
                if attr in obj.columns:
                    return name, obj, geo_col, attr

    return None, None, None, None


# ----------------------------------------------------------------------------
# 2. Localizar coluna de coorte
# ----------------------------------------------------------------------------

cohort_candidates = [
    "g_m_cs",
    "g_m",
    "cohort",
    "coorte",
    "cohort_cs",
    "first_treat",
    "first_treated",
    "first_treat_year",
    "ano_primeiro_tratamento",
    "ano_tratamento",
    "treatment_year",
]

src_name, src_df, geo_col, cohort_col = _find_first_valid_source(cohort_candidates)

if src_df is None:
    print("ERRO: não encontrei nenhuma coluna de coorte em DataFrames na memória.")
    print()
    print("DataFrames encontrados e respectivas colunas possíveis:")
    for name, obj in globals().items():
        if isinstance(obj, pd.DataFrame):
            cols = list(obj.columns)
            maybe = [c for c in cols if any(k in c.lower() for k in ["cohort", "coorte", "treat", "trat", "g_m"])]
            if maybe:
                print(f"- {name}: {maybe}")
    raise KeyError("Coluna de coorte não encontrada. Verifique onde está g_m_cs.")

print(f"Fonte usada para coorte: {src_name}")
print(f"Coluna municipal: {geo_col}")
print(f"Coluna de coorte: {cohort_col}")

# ----------------------------------------------------------------------------
# 3. Construir mapa município -> coorte
# ----------------------------------------------------------------------------

tmp_cohort = src_df[[geo_col, cohort_col]].copy()
tmp_cohort = tmp_cohort.rename(columns={geo_col: "geocode", cohort_col: "cohort"})

# Padronizar tipos para evitar problema int vs str
tmp_cohort["geocode_key"] = tmp_cohort["geocode"].astype(str)
tmp_cohort["cohort"] = pd.to_numeric(tmp_cohort["cohort"], errors="coerce")

# Manter apenas municípios comuns
common_keys = pd.Index(common_munis).astype(str)
tmp_cohort = tmp_cohort[tmp_cohort["geocode_key"].isin(common_keys)]

# Remover never treated/NaN se houver
tmp_cohort = tmp_cohort.dropna(subset=["cohort"])

# Se houver várias linhas por município, pega a primeira coorte observada
muni_cohort = (
    tmp_cohort
    .sort_values(["geocode_key", "cohort"])
    .groupby("geocode_key")["cohort"]
    .first()
)

# Aplicar ao b4m8_full
b4m8_full = b4m8_full.copy()
b4m8_full["geocode_key"] = b4m8_full.index.astype(str)
b4m8_full["cohort"] = b4m8_full["geocode_key"].map(muni_cohort)

n_missing = b4m8_full["cohort"].isna().sum()
print(f"Municípios em b4m8_full: {len(b4m8_full)}")
print(f"Municípios com coorte encontrada: {len(b4m8_full) - n_missing}")
print(f"Municípios sem coorte: {n_missing}")

if n_missing > 0:
    print()
    print("AVISO: há municípios sem coorte. Eles serão removidos da análise por coorte.")
    print("Primeiros municípios sem coorte:")
    print(list(b4m8_full.loc[b4m8_full["cohort"].isna(), "geocode_key"].head(10)))

# ----------------------------------------------------------------------------
# 4. KAYA por coorte
# ----------------------------------------------------------------------------

print()
print("=" * 78)
print("3. KAYA POR COORTE DE TRATAMENTO")
print("=" * 78)

b4m8_cohort = b4m8_full.dropna(subset=["cohort"]).copy()
b4m8_cohort["cohort"] = b4m8_cohort["cohort"].astype(int)

por_cohort = b4m8_cohort.groupby("cohort", observed=True).agg(
    n=("delta_intensity_kgco2_ton", "size"),
    int_pre_med=("intensity_pre", "median"),
    int_pos_med=("intensity_pos", "median"),
    delta_med=("delta_intensity_kgco2_ton", "median"),
    delta_rel_med=("delta_relativo_pct", "median"),
    delta_rel_mean=("delta_relativo_pct", "mean"),
    n_caiu=("delta_intensity_kgco2_ton", lambda x: int((x < 0).sum())),
    n_subiu=("delta_intensity_kgco2_ton", lambda x: int((x > 0).sum())),
    termo_escala_med=("termo_escala_tco2e", "median"),
    termo_int_med=("termo_intensidade_tco2e", "median"),
).reset_index()

print(f"{'cohort':>8}{'n':>5}{'int pré':>10}{'int pós':>10}"
      f"{'Δ med':>9}{'Δ rel%':>9}{'caiu':>7}{'subiu':>7}"
      f"{'escala':>11}{'intens.':>11}")
print("-" * 88)

for _, row in por_cohort.iterrows():
    print(f"{int(row['cohort']):>8}{int(row['n']):>5}"
          f"{row['int_pre_med']:>10.2f}{row['int_pos_med']:>10.2f}"
          f"{row['delta_med']:>+9.2f}{row['delta_rel_med']:>+8.1f}%"
          f"{int(row['n_caiu']):>7}{int(row['n_subiu']):>7}"
          f"{row['termo_escala_med']:>+11,.0f}{row['termo_int_med']:>+11,.0f}")

print()
print("INTERPRETAÇÃO:")
print("- Se todas ou quase todas as coortes têm Δ med negativo: queda robusta entre coortes.")
print("- Se n_caiu ≈ n em todas as coortes: queda generalizada, não concentrada em uma safra/coorte.")
print("- Se termo_escala é positivo e termo_intensidade negativo: expansão aumenta emissões totais,")
print("  mas intensidade kgCO2e/ton cai.")

# ----------------------------------------------------------------------------
# 5. KAYA por tercil share-cana, procurando tercil em qualquer painel disponível
# ----------------------------------------------------------------------------

print()
print("=" * 78)
print("4. KAYA POR TERCIL SHARE-CANA")
print("=" * 78)

tercil_candidates = [
    "tercil_share_cana",
    "tercil_cana",
    "share_cana_tercil",
    "tercil",
]

src_t_name, src_t_df, geo_t_col, tercil_col = _find_first_valid_source(tercil_candidates)

if src_t_df is None:
    print("AVISO: não encontrei coluna de tercil de share-cana em memória.")
    print("Pulando análise por tercil. Para incluir, rode o bloco B4.M.5 antes.")
else:
    print(f"Fonte usada para tercil: {src_t_name}")
    print(f"Coluna municipal: {geo_t_col}")
    print(f"Coluna de tercil: {tercil_col}")

    tmp_tercil = src_t_df[[geo_t_col, tercil_col]].copy()
    tmp_tercil = tmp_tercil.rename(columns={geo_t_col: "geocode", tercil_col: "tercil"})
    tmp_tercil["geocode_key"] = tmp_tercil["geocode"].astype(str)
    tmp_tercil = tmp_tercil[tmp_tercil["geocode_key"].isin(common_keys)]
    tmp_tercil = tmp_tercil.dropna(subset=["tercil"])

    muni_tercil = (
        tmp_tercil
        .groupby("geocode_key")["tercil"]
        .first()
    )

    b4m8_full["tercil"] = b4m8_full["geocode_key"].map(muni_tercil)

    b4m8_tercil = b4m8_full.dropna(subset=["tercil"]).copy()

    por_tercil = b4m8_tercil.groupby("tercil", observed=True).agg(
        n=("delta_intensity_kgco2_ton", "size"),
        int_pre_med=("intensity_pre", "median"),
        int_pos_med=("intensity_pos", "median"),
        delta_med=("delta_intensity_kgco2_ton", "median"),
        delta_rel_med=("delta_relativo_pct", "median"),
        n_caiu=("delta_intensity_kgco2_ton", lambda x: int((x < 0).sum())),
        n_subiu=("delta_intensity_kgco2_ton", lambda x: int((x > 0).sum())),
        termo_escala_med=("termo_escala_tco2e", "median"),
        termo_int_med=("termo_intensidade_tco2e", "median"),
    ).reset_index()

    print(f"{'tercil':<14}{'n':>5}{'int pré':>10}{'int pós':>10}"
          f"{'Δ med':>9}{'Δ rel%':>9}{'caiu':>7}{'subiu':>7}"
          f"{'escala':>11}{'intens.':>11}")
    print("-" * 92)

    for _, row in por_tercil.iterrows():
        print(f"{str(row['tercil']):<14}{int(row['n']):>5}"
              f"{row['int_pre_med']:>10.2f}{row['int_pos_med']:>10.2f}"
              f"{row['delta_med']:>+9.2f}{row['delta_rel_med']:>+8.1f}%"
              f"{int(row['n_caiu']):>7}{int(row['n_subiu']):>7}"
              f"{row['termo_escala_med']:>+11,.0f}{row['termo_int_med']:>+11,.0f}")

# ----------------------------------------------------------------------------
# 6. Salvar outputs
# ----------------------------------------------------------------------------

try:
    b4m8_full.to_csv(interim("b4m8_robustez_decomposicao.csv"))
    por_cohort.to_csv(interim("b4m8_robustez_por_coorte.csv"), index=False)
    if "por_tercil" in globals():
        por_tercil.to_csv(interim("b4m8_robustez_por_tercil.csv"), index=False)
    print()
    print("OK arquivos salvos via interim().")
except Exception as e:
    b4m8_full.to_csv("b4m8_robustez_decomposicao.csv")
    por_cohort.to_csv("b4m8_robustez_por_coorte.csv", index=False)
    if "por_tercil" in globals():
        por_tercil.to_csv("b4m8_robustez_por_tercil.csv", index=False)
    print()
    print(f"AVISO: interim() falhou ({type(e).__name__}: {e}).")
    print("Arquivos salvos no diretório atual.")

print()
print("=" * 78)
print("CONCLUSÃO PRELIMINAR A PARTIR DO QUE JÁ RODOU")
print("=" * 78)
print("Até a análise por decil, o achado está muito forte:")
print("- 193/193 municípios têm queda de intensidade.")
print("- O maior delta ainda é negativo: max = -0.630 kgCO2e/ton.")
print("- 0 municípios têm queda relativa perto de zero, usando |delta rel| < 1%.")
print("- Todos os decis de produção têm mediana negativa.")
print("- Portanto, a queda não parece ser puxada só pelos municípios gigantes.")

HOTFIX — localizar coluna de coorte e rodar KAYA por coorte
Fonte usada para coorte: panel
Coluna municipal: geocode
Coluna de coorte: g_m
Municípios em b4m8_full: 193
Municípios com coorte encontrada: 193
Municípios sem coorte: 0

3. KAYA POR COORTE DE TRATAMENTO
  cohort    n   int pré   int pós    Δ med   Δ rel%   caiu  subiu     escala    intens.
----------------------------------------------------------------------------------------
    2019    2     33.35     31.27    -2.07    -6.2%      2      0    +30,530     -9,148
    2020  142     23.58     22.23    -1.33    -5.7%    142      0     +9,339     -2,550
    2021   40     64.81     59.93    -4.02    -6.6%     40      0     +9,965     -3,244
    2022    7     92.07     84.75    -7.33    -7.4%      7      0     +2,993       -967
    2023    2     36.16     33.66    -2.50    -6.5%      2      0     +4,839     -1,619

INTERPRETAÇÃO:
- Se todas ou quase todas as coortes têm Δ med negativo: queda robusta entre coortes.
- Se n_caiu ≈ n 